In [1]:
!pip install beautifulsoup4 tokenizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.1/83.1 kB 3.4 MB/s eta 0:00:00


In [2]:
import requests
from bs4 import BeautifulSoup

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

In [4]:
urls = [
    "https://en.wikipedia.org/wiki/Artificial_intelligence",      # Artificial intelligence
    "https://en.wikipedia.org/wiki/Computer_vision",              # computer vision
    "https://en.wikipedia.org/wiki/Transformer_(deep_learning)"   # Transformers
    "https://en.wikipedia.org/wiki/Natural_language_processing"   # NLP
]

In [6]:
corpus = []

for url in urls:
  response = requests.get(url, headers=headers)
  b_soup = BeautifulSoup(response.text, "html.parser")

  for p in b_soup.find_all("p"):
    text = p.get_text(separator=" ", strip=True)
    if len(text) > 30:
      corpus.append(text)

print("Total Text Collected", len(corpus))

Total Text Collected 226


In [7]:
# write to text file
with open("corpus.txt", "w") as f:
  for text in corpus:
    f.write(text + "\n")

In [8]:
# clean text
import re

In [13]:
def clean_data(text):
  # Remove citations
  text = re.sub(r"\[[^\]]+\]", "", text)

  # Remove whitespaces
  text = re.sub(r"\s+", " ", text)

  return text.strip()

In [14]:
# cleaned corpus

cleaned_corpus = []

for line in corpus:
  cleaned_text = clean_data(line)
  if len(cleaned_text.split()) > 8:
    cleaned_corpus.append(cleaned_text)


In [16]:
# write to text file
with open("corpus_cleaned.txt", "w") as f:
  for text in cleaned_corpus:
    f.write(text + "\n")

In [17]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

In [18]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

In [19]:
trainer = BpeTrainer(
    vocab_size=5000,
    min_frequency=2,
    special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"]
)

In [21]:
tokenizer.train(["corpus_cleaned.txt"], trainer)

In [22]:
tokenizer.save("subword_tokenizer.json")

In [28]:
input = "Natural language processing (NLP) is the processing of natural language information by a computer"

In [29]:
encoded = tokenizer.encode(input)

In [30]:
print(encoded.tokens)

['N', 'atural', 'language', 'processing', '(', 'NLP', ')', 'is', 'the', 'processing', 'of', 'natural', 'language', 'information', 'by', 'a', 'computer']


In [23]:
input = "Cross-modal self-supervised pretraining enhanced electroencephalography-based emotion recognition."

In [24]:
encoded = tokenizer.encode(input)

In [27]:
print(encoded.tokens)

['C', 'ro', 's', 's', '-', 'mod', 'al', 'self', '-', 'supervised', 'pretraining', 'enh', 'an', 'ced', 'elect', 'ro', 'ence', 'ph', 'al', 'ograph', 'y', '-', 'based', 'emot', 'ion', 'recognition', '.']
